In [23]:
suppressPackageStartupMessages({
    library(dplyr)
    library(stringr)
    library(Biostrings)
    library(data.table)
    library(igraph)
    library(pwalign)
    library(Rsamtools)
    library(parallel)
})

source("/srv/home/mlef0011/VDARK/src/utils.r")

WD        <- "/srv/home/mlef0011/VDARK"
MINIMAP   <- "/srv/home/mlef0011/anaconda3/envs/VDARK/bin/minimap2"
REF       <- "/srv/home/mlef0011/rawdata/ref_genome/reference_genome_GRCh37.fa"
N_CORES   <- 20
K = 31
label = paste0("k",K)

### Load data

In [2]:
load_reads <- function(r1, r2) {
    fq1 <- readDNAStringSet(r1, format = "fastq")
    fq2 <- readDNAStringSet(r2, format = "fastq")
    data.frame(
        read_ID  = c(names(fq1), names(fq2)),
        sequence = as.character(c(fq1, fq2)),
        stringsAsFactors = FALSE
    )
}

tumour_reads <- load_reads(
    file.path(WD, paste0("rawdata/reads/tumour_R1_tumour_specific_",label,".fq")),
    file.path(WD, paste0("rawdata/reads/tumour_R2_tumour_specific_",label,".fq"))
)

### Associate each k-mer with its original read.

In [272]:
system("/srv/home/mlef0011/anaconda3/condabin/conda run -n VDARK python3 /srv/home/mlef0011/VDARK/src/build_read_kmer_table.py")

### Clustering reads sharing same k-mers to perfom local assembly

In [3]:
read_kmer_df <- fread(file.path(WD, paste0("rawdata/reads/read_kmer_association_",label,".tsv")))
read_kmer_df <- read_kmer_df[, if (.N <= 50) .SD, by = kmer]  # drop high-freq k-mers

pairs <- read_kmer_df[, {
    ids <- read_ID
    if (length(ids) >= 2) {
        idx <- combn(length(ids), 2)
        data.table(read_ID.x = ids[idx[1,]], read_ID.y = ids[idx[2,]])
    }
}, by = kmer][, .(weight = .N), by = .(read_ID.x, read_ID.y)]

G        <- graph_from_data_frame(pairs, directed = FALSE)
E(G)$weight <- pairs$weight
clusters <- get("components", envir = asNamespace("igraph"))(G)
cat(clusters$no, " clusters")

3235  clusters

In [4]:
read_ID <- "C23MCACXX130513:7:2110:4469:879864"
read_ID %in% read_kmer_df$read_ID
cluster_ID <- clusters$membership[read_ID]
cluster_ID = 10

[1] FALSE

In [5]:
logs <- character(0)
buf  <- function(...) logs <<- c(logs, paste0("[", format(Sys.time(), "%H:%M:%S"), "] ", ...))
k = K

### Construct tumor contig

In [6]:
tmp_cl         <- file.path(WD, "tmp", cluster_ID)
dir.create(tmp_cl, showWarnings = FALSE, recursive = TRUE)

reads_in  <- names(clusters$membership)[clusters$membership == cluster_ID]
kmers_sig <- unique(read_kmer_df$kmer[read_kmer_df$read_ID %in% reads_in])
all_kmers <- unique(unlist(lapply(
    tumour_reads$sequence[tumour_reads$read_ID %in% reads_in], function(seq) get_kmers(seq,K)
)))
cat("  ", length(reads_in), " reads | ", length(kmers_sig), " specific k-mers")

contig_tumour <- assemble_kmers(kmers_sig, k = k)
cat("  Tumour contig: ", nchar(contig_tumour[1]), " bp")
contig_tumour

   6  reads |  12  specific k-mers  Tumour contig:  42  bp

[1] "GTTTGACCGTAACTACTATTCACGAAAAAGATCTACTCATTA"
[2] "TAATGAGTAGATCTTTTTCGTGAATAGTAGTTACGGTCAAAC"

### Fetch normal reads via flanking k-mers 


In [7]:
flanking_kmers <- setdiff(all_kmers, kmers_sig)
flanking_kmers <- flanking_kmers[grepl("^[ACGT]+$", flanking_kmers)]

flanking_fa    <- file.path(tmp_cl, paste0("flanking_", cluster_ID, "_", label,".fa"))
kmc_db         <- file.path(tmp_cl, paste0("flanking_kmc", "_", label, cluster_ID))

writeLines(paste0(">k_", seq_along(flanking_kmers), "\n", flanking_kmers), flanking_fa)

system(paste(KMC, paste0("-k",K)," -t12 -ci1 -fm", flanking_fa, kmc_db, tmp_cl),
       ignore.stdout = TRUE, ignore.stderr = TRUE)

fq_R1_out <- file.path(tmp_cl, paste0("normal_R1_locus_", cluster_ID, "_", label , ".fq"))
fq_R2_out <- file.path(tmp_cl, paste0("normal_R2_locus_", cluster_ID, "_", label , ".fq"))

NORMAL_R1 = file.path(WD,"rawdata/reads/normal_R1.fq")
NORMAL_R2 = file.path(WD,"rawdata/reads/normal_R2.fq")

system(paste(KMC_TOOLS, "filter", kmc_db, "-ci1", NORMAL_R1, fq_R1_out),
       ignore.stdout = TRUE, ignore.stderr = TRUE)
system(paste(KMC_TOOLS, "filter", kmc_db, "-ci1", NORMAL_R2, fq_R2_out),
       ignore.stdout = TRUE, ignore.stderr = TRUE)

normal_df <- tryCatch(
    load_reads(fq_R1_out, fq_R2_out),
    error = function(e) data.frame(read_ID = character(), sequence = character())
)

# Precise filtering: keep reads with >= 30 flanking k-mer hits
if (nrow(normal_df) > 0) {
    pdict  <- PDict(unique(c(flanking_kmers,
                             as.character(reverseComplement(DNAStringSet(flanking_kmers))))))
    n_hits <- vapply(vwhichPDict(pdict, DNAStringSet(normal_df$sequence)),
                     length, integer(1))
    normal_df <- normal_df[n_hits >= 30, ]
}

if (nrow(normal_df) == 0) {
    cat("  No normal reads at locus -- skipping")
    message(paste(logs, collapse = "\n")); return(NULL)
}
cat("  Normal reads retained: ", nrow(normal_df))

  Normal reads retained:  90

### Assemble normal contig

In [8]:
k=K

In [9]:
kmer_counts      <- table(unlist(lapply(normal_df$sequence, function(seq) get_kmers(seq,k))))
cat("\n  Normal k-mers with cov>=10: ", sum(kmer_counts >= 10), " / ", length(kmer_counts))
kmer_to_assemble <- names(kmer_counts[kmer_counts >= 10])

if (length(kmer_to_assemble) == 0) {
    cat("\n  No k-mer with cov>=10 in normal -- skipping")
    message(paste(logs, collapse = "\n")); return(NULL)
}

contig_normal <- assemble_kmers(kmer_to_assemble, k = k)
if (is.null(contig_normal) || nchar(contig_normal[1]) < k) {
    cat("\n  Normal contig too short -- skipping")
    message(paste(logs, collapse = "\n")); return(NULL)
}
cat("\n  Normal contig: ", nchar(contig_normal[1]), " bp")



  Normal k-mers with cov>=10:  192  /  318
  Normal contig:  159  bp

### Alignment to retrieve SNV

In [10]:
aln <- best_alignment(contig_normal, contig_tumour)
format_alignment(aln, cluster_ID, buf = buf)

In [11]:
pat  <- strsplit(as.character(pattern(aln)), "")[[1]]
sub_ <- strsplit(as.character(subject(aln)), "")[[1]]
diff <- ifelse(pat == sub_, ".", "*")
print(paste0("  --- Alignment cluster ", cluster_ID, " ---"))
print(paste0("  Normal: ", paste(pat,  collapse = "")))
print(paste0("          ", paste(diff, collapse = "")))
print(paste0("  Tumour: ", paste(sub_, collapse = "")))

[1] "  --- Alignment cluster 10 ---"
[1] "  Normal: GTTTGACCGTAACTACTATTCATGAAAAAGATCTACTCATTA"
[1] "          ......................*..................."
[1] "  Tumour: GTTTGACCGTAACTACTATTCACGAAAAAGATCTACTCATTA"


In [31]:
mm <- mismatchTable(aln)
if (nrow(mm) == 0) {
    cat("  No mismatches detected")
    message(paste(logs, collapse = "\n")); return(NULL)
}
germline   <- is_germline(mm, contig_normal, fq_R1_out, fq_R2_out, cluster_ID, k=k, threshold = 3, buf = "")
mm_somatic <- mm[!germline, ]

if (nrow(mm_somatic) == 0) {
    cat("  No somatic variants (all germline)")
}
mm
mm_somatic
cat("\n  Somatic SNV(s): ", nrow(mm_somatic))


  No somatic variants (all germline)

PatternId,PatternStart,PatternEnd,PatternSubstring,SubjectStart,SubjectEnd,SubjectSubstring
<int>,<int>,<int>,<chr>,<int>,<int>,<chr>
1,95,95,T,23,23,C


PatternId,PatternStart,PatternEnd,PatternSubstring,SubjectStart,SubjectEnd,SubjectSubstring
<int>,<int>,<int>,<chr>,<int>,<int>,<chr>



  Somatic SNV(s):  0

In [29]:
writeLines(c(">contig_normal1", contig_normal[1]), 
           "/srv/home/mlef0011/VDARK/test/contigs_normal.fa")

In [268]:
# --- Lire le SAM ---
sam <- read.table("/srv/home/mlef0011/VDARK/test/contigs_normal_aln.sam", 
                  sep="\t", comment.char="@", fill=TRUE)
# Flag du read
flag <- sam[1, 2]
is_reverse <- bitwAnd(flag, 16) != 0  # TRUE si brin reverse
is_reverse
chrom <- sam[1, 3]
pos   <- sam[1, 4]  # position de début du contig sur le génome
contig = sam[1, 10]
# --- Boucle sur les mutations détectées ---
# Parser le soft-clip depuis le CIGAR
cigar <- as.character(sam[1, 6])

# Soft-clip gauche (S) ou hard-clip gauche (H)
left_clip <- 0
m <- regmatches(cigar, regexpr("^([0-9]+)[SH]", cigar))
if (length(m) > 0) left_clip <- as.integer(sub("[SH]", "", m))

# Soft-clip droit (S) ou hard-clip droit (H)
right_clip <- 0
m <- regmatches(cigar, regexpr("([0-9]+)[SH]$", cigar))
if (length(m) > 0) right_clip <- as.integer(sub("[SH]", "", m))
for (i in seq_len(nrow(mm_somatic))) {
    
    # Position dans le contig complet (pattern(best_aln) commence après le soft-clip)
    pos_in_contig <- mm_somatic$PatternStart[i]
    
    # Décalage : pos SAM = position génomique de la première base MAPPÉE
    # donc on soustrait le soft-clip pour avoir la position de la base 1 du contig
    pos_mut_genome <- if (is_reverse) {
        pos + nchar(as.character(contig)) - right_clip - pos_in_contig
    } else {
        pos - left_clip + pos_in_contig - 1
    }
    
    snv_ref <- as.character(mm_somatic$PatternSubstring[i])
    snv_alt <- as.character(mm_somatic$SubjectSubstring[i])
    
    if (is_reverse) {
        snv_ref <- as.character(reverseComplement(DNAString(snv_ref)))
        snv_alt <- as.character(reverseComplement(DNAString(snv_alt)))
    }
    
    cat("Chromosome :", as.character(chrom),
        "| Position :", pos_mut_genome,
        "| SNV :", snv_ref, "->", snv_alt, "\n")
}


[1] TRUE

Chromosome : 21 | Position : 10031113 | SNV : C -> T 
